# Embeddings Explorer: From Text to Vectors

This notebook is an interactive version of `scripts/check_embeddings_matrix.py`. It walks through how BERT converts text into dense vector representations (embeddings) and provides visualizations to build intuition about what those vectors look like.

## How Text Becomes Numbers

Language models cannot operate on raw text. The pipeline looks like this:

1. **Tokenization** -- The input string is split into sub-word tokens. BERT uses WordPiece tokenization, so uncommon words may be broken into pieces (e.g. "unhappiness" -> `["un", "##hap", "##pi", "##ness"]`). Two special tokens are added: `[CLS]` at the start and `[SEP]` at the end.

2. **Token IDs** -- Each token is mapped to an integer ID from BERT's 30,522-word vocabulary. This turns the sentence into a sequence of integers.

3. **Embedding lookup + Transformer layers** -- The token IDs index into a learned embedding table, producing an initial 768-dimensional vector per token. These vectors then pass through 12 Transformer encoder layers, where self-attention lets each token's representation absorb contextual information from every other token.

4. **Output** -- The result is a matrix of shape `(sequence_length, 768)`. Each row is a context-aware, dense vector for one token. These vectors encode meaning: semantically similar tokens end up close together in this 768-dimensional space.

Below, we will inspect each of these steps and visualize the resulting embedding space.

---
## 1. Install and Import Dependencies

In [ ]:
# Install dependencies (skip if already installed)
# %pip install torch transformers numpy matplotlib scikit-learn

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from transformers import BertTokenizer, BertModel
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")

---
## 2. Load BERT Tokenizer and Model

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME)
model.eval()  # inference mode

print(f"Model        : {MODEL_NAME}")
print(f"Vocab size   : {tokenizer.vocab_size:,}")
print(f"Hidden size  : {model.config.hidden_size}")
print(f"Layers       : {model.config.num_hidden_layers}")
print(f"Attn heads   : {model.config.num_attention_heads}")

---
## 3. Tokenize Your Input

Change `INPUT_TEXT` below to any sentence you like, then re-run the cells that follow to see how the embeddings change.

In [ ]:
# ============================================================
# CHANGE THIS to explore different sentences
# ============================================================
INPUT_TEXT = "I like my hotel room"
# ============================================================

# Tokenize
encoded = tokenizer(INPUT_TEXT, return_tensors="pt")
input_ids = encoded["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

print(f"Input text : {INPUT_TEXT}")
print(f"Num tokens : {len(tokens)} (including [CLS] and [SEP])\n")
print(f"{'Token':<15} {'ID':>8}")
print("-" * 25)
for tok, tid in zip(tokens, input_ids[0].tolist()):
    print(f"{tok:<15} {tid:>8}")

---
## 4. Generate Embeddings

In [ ]:
with torch.no_grad():
    outputs = model(**encoded)
    embeddings = outputs.last_hidden_state  # (1, seq_len, 768)

embeddings_np = embeddings.squeeze(0).numpy()  # (seq_len, 768)

print(f"Embedding tensor shape : {tuple(embeddings.shape)}")
print(f"  -> batch size        : {embeddings.shape[0]}")
print(f"  -> sequence length   : {embeddings.shape[1]}")
print(f"  -> hidden dimensions : {embeddings.shape[2]}")
print(f"\nEmbedding matrix (seq_len x hidden_size) : {embeddings_np.shape}")
print(f"Value range : [{embeddings_np.min():.4f}, {embeddings_np.max():.4f}]")
print(f"Mean        : {embeddings_np.mean():.6f}")
print(f"Std         : {embeddings_np.std():.4f}")

---
## 5. Visualization 1 -- Embedding Heatmap

Each row is a token and each column is one of the 768 hidden dimensions. We show the first 50 dimensions so the patterns are visible. Notice how `[CLS]` and `[SEP]` have distinctly different activation patterns from the content tokens.

In [ ]:
NUM_DIMS = 50  # number of dimensions to display

fig, ax = plt.subplots(figsize=(14, max(3, len(tokens) * 0.6)))
im = ax.imshow(embeddings_np[:, :NUM_DIMS], aspect="auto", cmap="viridis")

# Y-axis: token labels
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens, fontsize=11)

# X-axis: dimension indices
ax.set_xlabel("Hidden Dimension", fontsize=12)
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.set_title(f"Embedding Matrix  (tokens x first {NUM_DIMS} of {embeddings_np.shape[1]} dims)", fontsize=13)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Activation value", fontsize=11)

plt.tight_layout()
plt.show()

---
## 6. Visualization 2 -- Cosine Similarity Matrix

Cosine similarity measures the angle between two vectors, ignoring magnitude. Values close to 1 mean the model considers two tokens semantically related *in the current context*. Remember that BERT embeddings are contextual -- the same word will have different vectors in different sentences.

In [ ]:
cos_sim = cosine_similarity(embeddings_np)  # (seq_len, seq_len)

fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.9), max(5, len(tokens) * 0.8)))
im = ax.imshow(cos_sim, cmap="RdYlBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=11)
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens, fontsize=11)

# Annotate each cell with its value
for i in range(len(tokens)):
    for j in range(len(tokens)):
        color = "white" if abs(cos_sim[i, j]) > 0.6 else "black"
        ax.text(j, i, f"{cos_sim[i, j]:.2f}", ha="center", va="center",
                fontsize=9, color=color)

ax.set_title("Cosine Similarity Between Token Embeddings", fontsize=13)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Cosine similarity", fontsize=11)

plt.tight_layout()
plt.show()

---
## 7. Visualization 3 -- 2D Projection (PCA + t-SNE)

768 dimensions are impossible to visualize directly, but we can project them down to 2D. We show both PCA (preserves global variance) and t-SNE (preserves local neighborhood structure) side by side. Tokens that cluster together are considered similar by the model.

In [ ]:
n_tokens = len(tokens)

# PCA projection
pca = PCA(n_components=2)
pca_result = pca.fit_transform(embeddings_np)

# t-SNE projection
# Perplexity must be less than the number of samples
perplexity = min(5, n_tokens - 1)
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42,
            n_iter=1000, learning_rate="auto", init="pca")
tsne_result = tsne.fit_transform(embeddings_np)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, data, title in [
    (axes[0], pca_result, f"PCA  (explained var: {pca.explained_variance_ratio_.sum():.1%})"),
    (axes[1], tsne_result, "t-SNE"),
]:
    ax.scatter(data[:, 0], data[:, 1], s=120, c=range(n_tokens),
              cmap="tab10", edgecolors="black", linewidths=0.5, zorder=5)
    for i, tok in enumerate(tokens):
        ax.annotate(tok, (data[i, 0], data[i, 1]),
                    textcoords="offset points", xytext=(8, 6),
                    fontsize=11, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Token Embeddings Projected to 2D\n\"{INPUT_TEXT}\"", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Next Steps

Go back to **Section 3** and change `INPUT_TEXT` to a different sentence, then re-run all the cells below it. Here are some ideas to try:

- **Synonyms**: `"The cat sat on the mat"` vs. `"The feline sat on the rug"` -- do synonyms cluster together?
- **Polysemy**: `"I went to the bank to deposit money"` vs. `"I sat on the river bank"` -- does "bank" get different embeddings?
- **Longer text**: Try a full paragraph and watch how the heatmap and similarity matrix scale.
- **Technical text**: `"The gradient descent optimizer minimizes the loss function"` -- see how domain-specific tokens relate.

The key insight is that BERT embeddings are **contextual**: the same word produces different vectors depending on the surrounding sentence. This is what makes Transformer-based models so powerful compared to older static embeddings like Word2Vec.